<a href="https://colab.research.google.com/github/dyfzl/capstone/blob/yeon/%EC%BA%A1%EC%8A%A4%ED%86%A4_%EA%B0%90%EC%84%B1_%EB%B6%84%EB%A5%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#필요 패키지 설치
!pip install mxnet
!pip install gluonnlp pandas tqdm
!pip install sentencepiece
!pip install transformers
!pip install torch
!pip install pandas

In [ ]:
!pip uninstall numpy -y
!pip install numpy==1.23.5


In [ ]:
#kobert
!pip install 'git+https://github.com/SKTBrain/KoBERT.git#egg=kobert_tokenizer&subdirectory=kobert_hf'

  Cloning https://github.com/SKTBrain/KoBERT.git to /tmp/pip-install-nxka9umx/kobert-tokenizer_3b419a007d504fdebbe25270bfe05775
  Running command git clone --filter=blob:none --quiet https://github.com/SKTBrain/KoBERT.git /tmp/pip-install-nxka9umx/kobert-tokenizer_3b419a007d504fdebbe25270bfe05775
  Resolved https://github.com/SKTBrain/KoBERT.git to commit 5c46b1c68e4755b54879431bd302db621f4d2f47
  Preparing metadata (setup.py) ... done


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import gluonnlp as nlp
import numpy as np
from tqdm import tqdm, tqdm_notebook
import pandas as pd

In [ ]:
#  Hugging Face를 통한 모델 및 토크나이저 Import
from kobert_tokenizer import KoBERTTokenizer
from transformers import BertModel

from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup

In [ ]:
import torch

#GPU 설정
device = torch.device("cuda:0")

In [ ]:
import pandas as pd
import os
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
#데이터셋 로드
import pandas as pd
chatbot_data = pd.read_excel('/content/drive/MyDrive/캡스톤/한국어_단발성_대화_데이터셋.xlsx')

In [ ]:
chatbot_data.loc[(chatbot_data['Emotion'] == '공포'), 'Emotion'] = 0
chatbot_data.loc[(chatbot_data['Emotion'] == '슬픔'), 'Emotion'] = 1
chatbot_data.loc[(chatbot_data['Emotion'] == '중립'), 'Emotion'] = 2
chatbot_data.loc[(chatbot_data['Emotion'] == '놀람'), 'Emotion'] = 3
chatbot_data.loc[(chatbot_data['Emotion'] == '행복'), 'Emotion'] = 4
chatbot_data.loc[(chatbot_data['Emotion'] == '혐오'), 'Emotion'] = 5
chatbot_data.loc[(chatbot_data['Emotion'] == '분노'), 'Emotion'] = 6

In [ ]:
label_mapping = {'Positive': 0, 'Neutral': 1, 'Negative': 2}

# 기존 감정을 긍정/중립/부정으로 재분류 후 정수로 변환
chatbot_data.loc[(chatbot_data['Emotion'] == 4), 'Emotion'] = 'Positive'  # 행복
chatbot_data.loc[(chatbot_data['Emotion'].isin([2, 3])), 'Emotion'] = 'Neutral'  # 중립, 놀람
chatbot_data.loc[(chatbot_data['Emotion'].isin([0, 1, 5, 6])), 'Emotion'] = 'Negative'  # 공포, 슬픔, 혐오, 분노
chatbot_data['Emotion'] = chatbot_data['Emotion'].map(label_mapping)  # 문자열 → 정수로 매핑


In [ ]:
# 데이터셋 리스트 생성
data_list = []
for q, label in zip(chatbot_data['Sentence'], chatbot_data['Emotion']):
    data = []
    data.append(q)
    data.append(label)  # 정수로 저장
    data_list.append(data)

In [ ]:
from sklearn.model_selection import train_test_split
dataset_train, dataset_test = train_test_split(data_list, test_size=0.25, random_state=0)

print(len(dataset_train))
print(len(dataset_test))

28945
9649


In [ ]:
## Setting parameters
max_len = 64
batch_size = 64
warmup_ratio = 0.1
num_epochs = 5
max_grad_norm = 1
log_interval = 200
learning_rate =  5e-5

In [ ]:
class BERTSentenceTransform:
    r"""BERT style data transformation.

    Parameters
    ----------
    tokenizer : BERTTokenizer.
        Tokenizer for the sentences.
    max_seq_length : int.
        Maximum sequence length of the sentences.
    pad : bool, default True
        Whether to pad the sentences to maximum length.
    pair : bool, default True
        Whether to transform sentences or sentence pairs.
    """

    def __init__(self, tokenizer, max_seq_length,vocab, pad=True, pair=True):
        self._tokenizer = tokenizer
        self._max_seq_length = max_seq_length
        self._pad = pad
        self._pair = pair
        self._vocab = vocab

    def __call__(self, line):
        """Perform transformation for sequence pairs or single sequences.

        The transformation is processed in the following steps:
        - tokenize the input sequences
        - insert [CLS], [SEP] as necessary
        - generate type ids to indicate whether a token belongs to the first
        sequence or the second sequence.
        - generate valid length

        For sequence pairs, the input is a tuple of 2 strings:
        text_a, text_b.

        Inputs:
            text_a: 'is this jacksonville ?'
            text_b: 'no it is not'
        Tokenization:
            text_a: 'is this jack ##son ##ville ?'
            text_b: 'no it is not .'
        Processed:
            tokens: '[CLS] is this jack ##son ##ville ? [SEP] no it is not . [SEP]'
            type_ids: 0     0  0    0    0     0       0 0     1  1  1  1   1 1
            valid_length: 14

        For single sequences, the input is a tuple of single string:
        text_a.

        Inputs:
            text_a: 'the dog is hairy .'
        Tokenization:
            text_a: 'the dog is hairy .'
        Processed:
            text_a: '[CLS] the dog is hairy . [SEP]'
            type_ids: 0     0   0   0  0     0 0
            valid_length: 7

        Parameters
        ----------
        line: tuple of str
            Input strings. For sequence pairs, the input is a tuple of 2 strings:
            (text_a, text_b). For single sequences, the input is a tuple of single
            string: (text_a,).

        Returns
        -------
        np.array: input token ids in 'int32', shape (batch_size, seq_length)
        np.array: valid length in 'int32', shape (batch_size,)
        np.array: input token type ids in 'int32', shape (batch_size, seq_length)

        """

        # convert to unicode
        text_a = line[0]
        if self._pair:
            assert len(line) == 2
            text_b = line[1]

        tokens_a = self._tokenizer.tokenize(text_a)
        tokens_b = None

        if self._pair:
            tokens_b = self._tokenizer(text_b)

        if tokens_b:
            # Modifies `tokens_a` and `tokens_b` in place so that the total
            # length is less than the specified length.
            # Account for [CLS], [SEP], [SEP] with "- 3"
            self._truncate_seq_pair(tokens_a, tokens_b,
                                    self._max_seq_length - 3)
        else:
            # Account for [CLS] and [SEP] with "- 2"
            if len(tokens_a) > self._max_seq_length - 2:
                tokens_a = tokens_a[0:(self._max_seq_length - 2)]

        # The embedding vectors for `type=0` and `type=1` were learned during
        # pre-training and are added to the wordpiece embedding vector
        # (and position vector). This is not *strictly* necessary since
        # the [SEP] token unambiguously separates the sequences, but it makes
        # it easier for the model to learn the concept of sequences.

        # For classification tasks, the first vector (corresponding to [CLS]) is
        # used as as the "sentence vector". Note that this only makes sense because
        # the entire model is fine-tuned.
        #vocab = self._tokenizer.vocab
        vocab = self._vocab
        tokens = []
        tokens.append(vocab.cls_token)
        tokens.extend(tokens_a)
        tokens.append(vocab.sep_token)
        segment_ids = [0] * len(tokens)

        if tokens_b:
            tokens.extend(tokens_b)
            tokens.append(vocab.sep_token)
            segment_ids.extend([1] * (len(tokens) - len(segment_ids)))

        input_ids = self._tokenizer.convert_tokens_to_ids(tokens)

        # The valid length of sentences. Only real  tokens are attended to.
        valid_length = len(input_ids)

        if self._pad:
            # Zero-pad up to the sequence length.
            padding_length = self._max_seq_length - valid_length
            # use padding tokens for the rest
            input_ids.extend([vocab[vocab.padding_token]] * padding_length)
            segment_ids.extend([0] * padding_length)

        return np.array(input_ids, dtype='int32'), np.array(valid_length, dtype='int32'),\
            np.array(segment_ids, dtype='int32')

In [ ]:
from kobert_tokenizer import KoBERTTokenizer
from transformers import BertModel
from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup

class BERTDataset(Dataset):
    def __init__(self, dataset, sent_idx, label_idx, bert_tokenizer, vocab, max_len,
                 pad, pair):
        transform = BERTSentenceTransform(bert_tokenizer, max_seq_length=max_len, vocab=vocab, pad=pad, pair=pair)
        self.sentences = [transform([i[sent_idx]]) for i in dataset]
        self.labels = [np.int32(i[label_idx]) for i in dataset]

    def __getitem__(self, i):
        return (self.sentences[i] + (self.labels[i], ))

    def __len__(self):
        return (len(self.labels))

tokenizer = KoBERTTokenizer.from_pretrained('skt/kobert-base-v1')
bertmodel = BertModel.from_pretrained('skt/kobert-base-v1', return_dict=False)
vocab = nlp.vocab.BERTVocab.from_sentencepiece(tokenizer.vocab_file, padding_token='[PAD]')


data_train = BERTDataset(dataset_train, 0, 1, tokenizer, vocab, max_len, True, False)
data_test = BERTDataset(dataset_test, 0, 1, tokenizer, vocab, max_len, True, False)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.


In [ ]:
train_dataloader = torch.utils.data.DataLoader(data_train, batch_size=batch_size, num_workers=5)
test_dataloader = torch.utils.data.DataLoader(data_test, batch_size=batch_size, num_workers=5)

In [ ]:
# 데이터 로더 확인
print(data_train[0])  # 첫 번째 데이터 확인

(array([   2, 1189,  517, 6188, 7245, 7063,  517,  463, 6999, 6122, 7836,
       5966, 1698,  517, 6188, 7245, 7063,  463, 5561, 6398, 7870, 1801,
       6885, 7088, 5966, 1698, 5837, 5837,  517, 6188, 7245, 6398, 6037,
       7063,  463,  463,  364,  364,    3,    1,    1,    1,    1,    1,
          1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
          1,    1,    1,    1,    1,    1,    1,    1,    1], dtype=int32), array(39, dtype=int32), array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int32), 0)


KoBERT model 정의

In [ ]:
class BERTClassifier(nn.Module):
    def __init__(self,
                 bert,
                 hidden_size = 768,
                 num_classes=8,
                 dr_rate=None,
                 params=None):
        super(BERTClassifier, self).__init__()
        self.bert = bert
        self.dr_rate = dr_rate

        self.classifier = nn.Linear(hidden_size , num_classes)
        if dr_rate:
            self.dropout = nn.Dropout(p=dr_rate)

    def gen_attention_mask(self, token_ids, valid_length):
        attention_mask = torch.zeros_like(token_ids)
        for i, v in enumerate(valid_length):
            attention_mask[i][:v] = 1
        return attention_mask.float()

    def forward(self, token_ids, valid_length, segment_ids):
        attention_mask = self.gen_attention_mask(token_ids, valid_length)

        _, pooler = self.bert(input_ids = token_ids, token_type_ids = segment_ids.long(), attention_mask = attention_mask.float().to(token_ids.device))
        if self.dr_rate:
            out = self.dropout(pooler)
        return self.classifier(out)


In [ ]:
#정의한 모델 불러오기
model = BERTClassifier(bertmodel,  dr_rate=0.5).to(device)
#model = BERTClassifier(bertmodel,  dr_rate=0.5).to('cpu')

# Prepare optimizer and schedule (linear warmup and decay)
no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
]
optimizer = AdamW(optimizer_grouped_parameters, lr=learning_rate)
loss_fn = nn.CrossEntropyLoss()
t_total = len(train_dataloader) * num_epochs
warmup_step = int(t_total * warmup_ratio)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_step, num_training_steps=t_total)
def calc_accuracy(X,Y):
    max_vals, max_indices = torch.max(X, 1)
    train_acc = (max_indices == Y).sum().data.cpu().numpy()/max_indices.size()[0]
    return train_acc
train_dataloader

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:
for e in range(num_epochs):
    train_acc = 0.0
    test_acc = 0.0
    model.train()
    for batch_id, (token_ids, valid_length, segment_ids, label) in enumerate(tqdm_notebook(train_dataloader)):
        optimizer.zero_grad()
        token_ids = token_ids.long().to(device)
        segment_ids = segment_ids.long().to(device)
        valid_length= valid_length
        label = label.long().to(device)
        out = model(token_ids, valid_length, segment_ids)
        loss = loss_fn(out, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        scheduler.step()  # Update learning rate schedule
        train_acc += calc_accuracy(out, label)
        if batch_id % log_interval == 0:
            print("epoch {} batch id {} loss {} train acc {}".format(e+1, batch_id+1, loss.data.cpu().numpy(), train_acc / (batch_id+1)))
    print("epoch {} train acc {}".format(e+1, train_acc / (batch_id+1)))

    model.eval()
    for batch_id, (token_ids, valid_length, segment_ids, label) in enumerate(tqdm_notebook(test_dataloader)):
        token_ids = token_ids.long().to(device)
        segment_ids = segment_ids.long().to(device)
        valid_length= valid_length
        label = label.long().to(device)
        out = model(token_ids, valid_length, segment_ids)
        test_acc += calc_accuracy(out, label)
    print("epoch {} test acc {}".format(e+1, test_acc / (batch_id+1)))

<ipython-input-74-c43874aa7363>:5: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for batch_id, (token_ids, valid_length, segment_ids, label) in enumerate(tqdm_notebook(train_dataloader)):


  0%|          | 0/453 [00:00<?, ?it/s]

epoch 1 batch id 1 loss 2.0520737171173096 train acc 0.203125
epoch 1 batch id 201 loss 0.7200098037719727 train acc 0.5481187810945274
epoch 1 batch id 401 loss 0.7173854112625122 train acc 0.6300654613466334
epoch 1 train acc 0.6419519380599922


<ipython-input-74-c43874aa7363>:23: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for batch_id, (token_ids, valid_length, segment_ids, label) in enumerate(tqdm_notebook(test_dataloader)):


  0%|          | 0/151 [00:00<?, ?it/s]

...

epoch 4 test acc 0.7440236856331937


  0%|          | 0/453 [00:00<?, ?it/s]

epoch 5 batch id 1 loss 0.16888250410556793 train acc 0.9375
epoch 5 batch id 201 loss 0.2757667601108551 train acc 0.9270055970149254
epoch 5 batch id 401 loss 0.1797681748867035 train acc 0.9399158354114713
epoch 5 train acc 0.9401559050772627


  0%|          | 0/151 [00:00<?, ?it/s]

epoch 5 test acc 0.7487836194080281


In [ ]:
#토큰화
tokenizer = KoBERTTokenizer.from_pretrained('skt/kobert-base-v1')
#tok = tokenizer.tokenize

def predict(predict_sentence):

    data = [predict_sentence, '0']
    dataset_another = [data]

    another_test = BERTDataset(dataset_another, 0, 1, tokenizer, vocab, max_len, True, False)
    test_dataloader = torch.utils.data.DataLoader(another_test, batch_size=batch_size, num_workers=5)

    model.eval()

    for batch_id, (token_ids, valid_length, segment_ids, label) in enumerate(test_dataloader):
        token_ids = token_ids.long().to(device)
        segment_ids = segment_ids.long().to(device)

        valid_length= valid_length
        label = label.long().to(device)

        out = model (token_ids, valid_length, segment_ids)

        test_eval=[]
        for i in out:
            logits=i
            logits = logits.detach().cpu().numpy()

            if np.argmax(logits) == 0:
              test_eval.append("공포")
            elif np.argmax(logits) == 1:
              test_eval.append("슬")
            elif np.argmax(logits) == 2:
              test_eval.append("중립")
            elif np.argmax(logits) == 3:
              test_eval.append("놀")
            elif np.argmax(logits) == 4:
              test_eval.append("행복")
            elif np.argmax(logits) == 5:
              test_eval.append("혐오")
            elif np.argmax(logits) == 6:
              test_eval.append("분노")

            print(test_eval[0] + " 하루를 보냈네요.")
#질문 무한반복하기! 0 입력시 종료
end = 1
while end == 1 :
    sentence = input("하고싶은 말을 입력해주세요 : ")
    if sentence == 0 :
        break
    predict(sentence)
    print("\n")

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.


하고싶은 말을 입력해주세요 : lx정보연구원 정말 유익한 정보를 주네요
행복 하루를 보냈네요.


하고싶은 말을 입력해주세요 : lx 이렇게 안 봤는데.. 내부 조사가 필요해보입니다
중립 하루를 보냈네요.


하고싶은 말을 입력해주세요 : 아니 니네는 세금만 받아 처먹고 이게 뭐하는짓이야
분노 하루를 보냈네요.


하고싶은 말을 입력해주세요 : 집에가고싶다
슬 하루를 보냈네요.


하고싶은 말을 입력해주세요 : 앞으로도 파이팅입니다
행복 하루를 보냈네요.




KeyboardInterrupt: Interrupted by user

In [ ]:
#필요 패키지 설치
!pip install mxnet
!pip install gluonnlp pandas tqdm
!pip install sentencepiece
!pip install transformers
!pip install torch
!pip install pandas

In [ ]:
!pip uninstall numpy -y
!pip install numpy==1.23.5


In [ ]:
#kobert
!pip install 'git+https://github.com/SKTBrain/KoBERT.git#egg=kobert_tokenizer&subdirectory=kobert_hf'

  Cloning https://github.com/SKTBrain/KoBERT.git to /tmp/pip-install-nxka9umx/kobert-tokenizer_3b419a007d504fdebbe25270bfe05775
  Running command git clone --filter=blob:none --quiet https://github.com/SKTBrain/KoBERT.git /tmp/pip-install-nxka9umx/kobert-tokenizer_3b419a007d504fdebbe25270bfe05775
  Resolved https://github.com/SKTBrain/KoBERT.git to commit 5c46b1c68e4755b54879431bd302db621f4d2f47
  Preparing metadata (setup.py) ... done


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import gluonnlp as nlp
import numpy as np
from tqdm import tqdm, tqdm_notebook
import pandas as pd

In [ ]:
#  Hugging Face를 통한 모델 및 토크나이저 Import
from kobert_tokenizer import KoBERTTokenizer
from transformers import BertModel

from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup

In [ ]:
import torch

#GPU 설정
device = torch.device("cuda:0")

In [ ]:
import pandas as pd
import os
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
#데이터셋 로드
import pandas as pd
chatbot_data = pd.read_excel('/content/drive/MyDrive/캡스톤/한국어_단발성_대화_데이터셋.xlsx')

In [ ]:
chatbot_data.loc[(chatbot_data['Emotion'] == '공포'), 'Emotion'] = 0
chatbot_data.loc[(chatbot_data['Emotion'] == '슬픔'), 'Emotion'] = 1
chatbot_data.loc[(chatbot_data['Emotion'] == '중립'), 'Emotion'] = 2
chatbot_data.loc[(chatbot_data['Emotion'] == '놀람'), 'Emotion'] = 3
chatbot_data.loc[(chatbot_data['Emotion'] == '행복'), 'Emotion'] = 4
chatbot_data.loc[(chatbot_data['Emotion'] == '혐오'), 'Emotion'] = 5
chatbot_data.loc[(chatbot_data['Emotion'] == '분노'), 'Emotion'] = 6

In [ ]:
label_mapping = {'Positive': 0, 'Neutral': 1, 'Negative': 2}

# 기존 감정을 긍정/중립/부정으로 재분류 후 정수로 변환
chatbot_data.loc[(chatbot_data['Emotion'] == 4), 'Emotion'] = 'Positive'  # 행복
chatbot_data.loc[(chatbot_data['Emotion'].isin([2, 3])), 'Emotion'] = 'Neutral'  # 중립, 놀람
chatbot_data.loc[(chatbot_data['Emotion'].isin([0, 1, 5, 6])), 'Emotion'] = 'Negative'  # 공포, 슬픔, 혐오, 분노
chatbot_data['Emotion'] = chatbot_data['Emotion'].map(label_mapping)  # 문자열 → 정수로 매핑


In [ ]:
# 데이터셋 리스트 생성
data_list = []
for q, label in zip(chatbot_data['Sentence'], chatbot_data['Emotion']):
    data = []
    data.append(q)
    data.append(label)  # 정수로 저장
    data_list.append(data)

In [ ]:
from sklearn.model_selection import train_test_split
dataset_train, dataset_test = train_test_split(data_list, test_size=0.25, random_state=0)

print(len(dataset_train))
print(len(dataset_test))

28945
9649


In [ ]:
## Setting parameters
max_len = 64
batch_size = 64
warmup_ratio = 0.1
num_epochs = 5
max_grad_norm = 1
log_interval = 200
learning_rate =  5e-5

In [ ]:
class BERTSentenceTransform:
    r"""BERT style data transformation.

    Parameters
    ----------
    tokenizer : BERTTokenizer.
        Tokenizer for the sentences.
    max_seq_length : int.
        Maximum sequence length of the sentences.
    pad : bool, default True
        Whether to pad the sentences to maximum length.
    pair : bool, default True
        Whether to transform sentences or sentence pairs.
    """

    def __init__(self, tokenizer, max_seq_length,vocab, pad=True, pair=True):
        self._tokenizer = tokenizer
        self._max_seq_length = max_seq_length
        self._pad = pad
        self._pair = pair
        self._vocab = vocab

    def __call__(self, line):
        """Perform transformation for sequence pairs or single sequences.

        The transformation is processed in the following steps:
        - tokenize the input sequences
        - insert [CLS], [SEP] as necessary
        - generate type ids to indicate whether a token belongs to the first
        sequence or the second sequence.
        - generate valid length

        For sequence pairs, the input is a tuple of 2 strings:
        text_a, text_b.

        Inputs:
            text_a: 'is this jacksonville ?'
            text_b: 'no it is not'
        Tokenization:
            text_a: 'is this jack ##son ##ville ?'
            text_b: 'no it is not .'
        Processed:
            tokens: '[CLS] is this jack ##son ##ville ? [SEP] no it is not . [SEP]'
            type_ids: 0     0  0    0    0     0       0 0     1  1  1  1   1 1
            valid_length: 14

        For single sequences, the input is a tuple of single string:
        text_a.

        Inputs:
            text_a: 'the dog is hairy .'
        Tokenization:
            text_a: 'the dog is hairy .'
        Processed:
            text_a: '[CLS] the dog is hairy . [SEP]'
            type_ids: 0     0   0   0  0     0 0
            valid_length: 7

        Parameters
        ----------
        line: tuple of str
            Input strings. For sequence pairs, the input is a tuple of 2 strings:
            (text_a, text_b). For single sequences, the input is a tuple of single
            string: (text_a,).

        Returns
        -------
        np.array: input token ids in 'int32', shape (batch_size, seq_length)
        np.array: valid length in 'int32', shape (batch_size,)
        np.array: input token type ids in 'int32', shape (batch_size, seq_length)

        """

        # convert to unicode
        text_a = line[0]
        if self._pair:
            assert len(line) == 2
            text_b = line[1]

        tokens_a = self._tokenizer.tokenize(text_a)
        tokens_b = None

        if self._pair:
            tokens_b = self._tokenizer(text_b)

        if tokens_b:
            # Modifies `tokens_a` and `tokens_b` in place so that the total
            # length is less than the specified length.
            # Account for [CLS], [SEP], [SEP] with "- 3"
            self._truncate_seq_pair(tokens_a, tokens_b,
                                    self._max_seq_length - 3)
        else:
            # Account for [CLS] and [SEP] with "- 2"
            if len(tokens_a) > self._max_seq_length - 2:
                tokens_a = tokens_a[0:(self._max_seq_length - 2)]

        # The embedding vectors for `type=0` and `type=1` were learned during
        # pre-training and are added to the wordpiece embedding vector
        # (and position vector). This is not *strictly* necessary since
        # the [SEP] token unambiguously separates the sequences, but it makes
        # it easier for the model to learn the concept of sequences.

        # For classification tasks, the first vector (corresponding to [CLS]) is
        # used as as the "sentence vector". Note that this only makes sense because
        # the entire model is fine-tuned.
        #vocab = self._tokenizer.vocab
        vocab = self._vocab
        tokens = []
        tokens.append(vocab.cls_token)
        tokens.extend(tokens_a)
        tokens.append(vocab.sep_token)
        segment_ids = [0] * len(tokens)

        if tokens_b:
            tokens.extend(tokens_b)
            tokens.append(vocab.sep_token)
            segment_ids.extend([1] * (len(tokens) - len(segment_ids)))

        input_ids = self._tokenizer.convert_tokens_to_ids(tokens)

        # The valid length of sentences. Only real  tokens are attended to.
        valid_length = len(input_ids)

        if self._pad:
            # Zero-pad up to the sequence length.
            padding_length = self._max_seq_length - valid_length
            # use padding tokens for the rest
            input_ids.extend([vocab[vocab.padding_token]] * padding_length)
            segment_ids.extend([0] * padding_length)

        return np.array(input_ids, dtype='int32'), np.array(valid_length, dtype='int32'),\
            np.array(segment_ids, dtype='int32')

In [ ]:
from kobert_tokenizer import KoBERTTokenizer
from transformers import BertModel
from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup

class BERTDataset(Dataset):
    def __init__(self, dataset, sent_idx, label_idx, bert_tokenizer, vocab, max_len,
                 pad, pair):
        transform = BERTSentenceTransform(bert_tokenizer, max_seq_length=max_len, vocab=vocab, pad=pad, pair=pair)
        self.sentences = [transform([i[sent_idx]]) for i in dataset]
        self.labels = [np.int32(i[label_idx]) for i in dataset]

    def __getitem__(self, i):
        return (self.sentences[i] + (self.labels[i], ))

    def __len__(self):
        return (len(self.labels))

tokenizer = KoBERTTokenizer.from_pretrained('skt/kobert-base-v1')
bertmodel = BertModel.from_pretrained('skt/kobert-base-v1', return_dict=False)
vocab = nlp.vocab.BERTVocab.from_sentencepiece(tokenizer.vocab_file, padding_token='[PAD]')


data_train = BERTDataset(dataset_train, 0, 1, tokenizer, vocab, max_len, True, False)
data_test = BERTDataset(dataset_test, 0, 1, tokenizer, vocab, max_len, True, False)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.


In [ ]:
train_dataloader = torch.utils.data.DataLoader(data_train, batch_size=batch_size, num_workers=5)
test_dataloader = torch.utils.data.DataLoader(data_test, batch_size=batch_size, num_workers=5)

In [ ]:
# 데이터 로더 확인
print(data_train[0])  # 첫 번째 데이터 확인

(array([   2, 1189,  517, 6188, 7245, 7063,  517,  463, 6999, 6122, 7836,
       5966, 1698,  517, 6188, 7245, 7063,  463, 5561, 6398, 7870, 1801,
       6885, 7088, 5966, 1698, 5837, 5837,  517, 6188, 7245, 6398, 6037,
       7063,  463,  463,  364,  364,    3,    1,    1,    1,    1,    1,
          1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
          1,    1,    1,    1,    1,    1,    1,    1,    1], dtype=int32), array(39, dtype=int32), array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int32), 0)


KoBERT model 정의

In [ ]:
class BERTClassifier(nn.Module):
    def __init__(self,
                 bert,
                 hidden_size = 768,
                 num_classes=8,
                 dr_rate=None,
                 params=None):
        super(BERTClassifier, self).__init__()
        self.bert = bert
        self.dr_rate = dr_rate

        self.classifier = nn.Linear(hidden_size , num_classes)
        if dr_rate:
            self.dropout = nn.Dropout(p=dr_rate)

    def gen_attention_mask(self, token_ids, valid_length):
        attention_mask = torch.zeros_like(token_ids)
        for i, v in enumerate(valid_length):
            attention_mask[i][:v] = 1
        return attention_mask.float()

    def forward(self, token_ids, valid_length, segment_ids):
        attention_mask = self.gen_attention_mask(token_ids, valid_length)

        _, pooler = self.bert(input_ids = token_ids, token_type_ids = segment_ids.long(), attention_mask = attention_mask.float().to(token_ids.device))
        if self.dr_rate:
            out = self.dropout(pooler)
        return self.classifier(out)


In [ ]:
#정의한 모델 불러오기
model = BERTClassifier(bertmodel,  dr_rate=0.5).to(device)
#model = BERTClassifier(bertmodel,  dr_rate=0.5).to('cpu')

# Prepare optimizer and schedule (linear warmup and decay)
no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
]
optimizer = AdamW(optimizer_grouped_parameters, lr=learning_rate)
loss_fn = nn.CrossEntropyLoss()
t_total = len(train_dataloader) * num_epochs
warmup_step = int(t_total * warmup_ratio)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_step, num_training_steps=t_total)
def calc_accuracy(X,Y):
    max_vals, max_indices = torch.max(X, 1)
    train_acc = (max_indices == Y).sum().data.cpu().numpy()/max_indices.size()[0]
    return train_acc
train_dataloader

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:
for e in range(num_epochs):
    train_acc = 0.0
    test_acc = 0.0
    model.train()
    for batch_id, (token_ids, valid_length, segment_ids, label) in enumerate(tqdm_notebook(train_dataloader)):
        optimizer.zero_grad()
        token_ids = token_ids.long().to(device)
        segment_ids = segment_ids.long().to(device)
        valid_length= valid_length
        label = label.long().to(device)
        out = model(token_ids, valid_length, segment_ids)
        loss = loss_fn(out, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        scheduler.step()  # Update learning rate schedule
        train_acc += calc_accuracy(out, label)
        if batch_id % log_interval == 0:
            print("epoch {} batch id {} loss {} train acc {}".format(e+1, batch_id+1, loss.data.cpu().numpy(), train_acc / (batch_id+1)))
    print("epoch {} train acc {}".format(e+1, train_acc / (batch_id+1)))

    model.eval()
    for batch_id, (token_ids, valid_length, segment_ids, label) in enumerate(tqdm_notebook(test_dataloader)):
        token_ids = token_ids.long().to(device)
        segment_ids = segment_ids.long().to(device)
        valid_length= valid_length
        label = label.long().to(device)
        out = model(token_ids, valid_length, segment_ids)
        test_acc += calc_accuracy(out, label)
    print("epoch {} test acc {}".format(e+1, test_acc / (batch_id+1)))

<ipython-input-75-c43874aa7363>:5: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for batch_id, (token_ids, valid_length, segment_ids, label) in enumerate(tqdm_notebook(train_dataloader)):


  0%|          | 0/453 [00:00<?, ?it/s]

epoch 1 batch id 1 loss 0.1251285821199417 train acc 0.953125
epoch 1 batch id 201 loss 0.26473724842071533 train acc 0.9407649253731343
epoch 1 batch id 401 loss 0.14985105395317078 train acc 0.948916770573566
epoch 1 train acc 0.9484340507726269


<ipython-input-75-c43874aa7363>:23: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for batch_id, (token_ids, valid_length, segment_ids, label) in enumerate(tqdm_notebook(test_dataloader)):


  0%|          | 0/151 [00:00<?, ?it/s]

...

epoch 4 test acc 0.7375764461413705


  0%|          | 0/453 [00:00<?, ?it/s]

epoch 5 batch id 1 loss 0.1330832540988922 train acc 0.953125
epoch 5 batch id 201 loss 0.11266554147005081 train acc 0.9457400497512438
epoch 5 batch id 401 loss 0.22682718932628632 train acc 0.9465399002493765
epoch 5 train acc 0.9451227924944813


  0%|          | 0/151 [00:00<?, ?it/s]

epoch 5 test acc 0.734949401946209


In [ ]:
!pip install flask flask-cors


In [ ]:
# 긍정적인 키워드 기반 감정 감지 로직 추가
def keyword_based_sentiment(sentence):
    positive_keywords = ["사랑", "좋아", "행복", "감사", "고마워"]
    for keyword in positive_keywords:
        if keyword in sentence:
            return "Positive"
    return None


In [ ]:
import pandas as pd
import re
import torch
from torch.utils.data import DataLoader

# 모델 관련 설정 (사전에 정의된 변수)
label_mapping = {"Positive": 0, "Neutral": 1, "Negative": 2}
emotion_mapping = {0: "공포", 1: "슬픔", 2: "중립", 3: "놀람", 4: "행복", 5: "혐오", 6: "분노"}
emotion_to_sentiment = {
    0: "Negative",  # 공포
    1: "Negative",  # 슬픔
    2: "Neutral",   # 중립
    3: "Neutral",   # 놀람
    4: "Positive",  # 행복
    5: "Negative",  # 혐오
    6: "Negative",  # 분노
}
max_len = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 특수 기호 및 이모지 제거 함수
def clean_text(text):
    # 한글, 영어, 공백만 남기고 나머지 제거
    text = re.sub(r"[^\uAC00-\uD7A3a-zA-Z\s]", "", text)  # 한글, 영어, 공백만 남김
    text = re.sub(r"\s+", " ", text)  # 연속된 공백 제거
    return text.strip()  # 앞뒤 공백 제거

# 키워드 기반 긍정 판단
def keyword_based_sentiment(sentence):
    positive_keywords = ["사랑", "좋아", "행복", "감사", "고마워"]
    for keyword in positive_keywords:
        if keyword in sentence:
            return "행복", "Positive"  # Original_Emotion, Sentiment
    return None, None

# 감정 분석 함수
def analyze_emotions(file_path):
    # 파일 읽기
    df = pd.read_csv(file_path)

    # "Comment" 열에서 문장 리스트 가져오기
    df['Comment'] = df['Comment'].fillna('').astype(str)  # 결측치 처리
    sentences = df['Comment'].apply(clean_text).tolist()  # 전처리 후 리스트로 변환

    original_emotions = []  # 원래 감정 저장 (7가지)
    sentiments = []  # 긍정, 중립, 부정 저장

    # 데이터 변환 및 감정 분석
    for sentence in sentences:
        # 키워드 기반 감정 판단
        keyword_emotion, keyword_sentiment = keyword_based_sentiment(sentence)
        if keyword_emotion and keyword_sentiment:
            original_emotions.append(keyword_emotion)
            sentiments.append(keyword_sentiment)
            continue  # 모델 분석을 건너뜀

        # 모델 기반 감정 분석
        data = [sentence, 0]  # 임시 라벨
        dataset = [data]

        test_dataset = BERTDataset(dataset, 0, 1, tokenizer, vocab, max_len, True, False)
        test_dataloader = DataLoader(test_dataset, batch_size=1, num_workers=0)

        model.eval()
        with torch.no_grad():
            for token_ids, valid_length, segment_ids, label in test_dataloader:
                token_ids = token_ids.long().to(device)
                segment_ids = segment_ids.long().to(device)
                out = model(token_ids, valid_length, segment_ids)
                logits = out.detach().cpu().numpy()
                pred_label = np.argmax(logits)

                # 세부 감정 및 긍정/중립/부정
                original_emotions.append(emotion_mapping[pred_label])
                sentiments.append(emotion_to_sentiment[pred_label])

    # 새로운 열 추가
    df['Original_Emotion'] = original_emotions  # 세부 감정
    df['Sentiment'] = sentiments  # 긍정/중립/부정

    # 결과 저장
    output_file = file_path.replace(".csv", "_with_emotions.csv")
    df.to_csv(output_file, index=False, encoding='utf-8-sig')

    print(f"분석 결과가 저장되었습니다: {output_file}")
    return output_file

# 실행 코드
if __name__ == "__main__":
    # CSV 파일 경로
    file_path = "/content/drive/MyDrive/캡스톤/youtube_comments.csv"  # 입력 파일 경로
    analyze_emotions(file_path)


분석 결과가 저장되었습니다: /content/drive/MyDrive/캡스톤/youtube_comments_with_emotions.csv


In [ ]:
import pandas as pd
import re

# 특수 기호 및 이모지 제거 함수
def clean_text(text):
    # 한글, 영어, 공백만 남기고 나머지 제거
    text = re.sub(r"[^\uAC00-\uD7A3a-zA-Z\s]", "", text)  # 한글, 영어, 공백만 남김
    text = re.sub(r"\s+", " ", text)  # 연속된 공백 제거
    return text.strip()  # 앞뒤 공백 제거

# 전처리 결과 출력 함수
def print_preprocessed_data(file_path):
    # 파일 읽기
    df = pd.read_csv(file_path)

    # "Comment" 열에서 결측치 처리 후 전처리 적용
    df['Comment'] = df['Comment'].fillna('').astype(str)  # 결측치 처리
    df['Cleaned_Comment'] = df['Comment'].apply(clean_text)  # 전처리

    # 원본과 전처리된 데이터 출력
    for original, cleaned in zip(df['Comment'], df['Cleaned_Comment']):
        print(f"원본: {original}")
        print(f"전처리: {cleaned}")
        print("-" * 50)

# 실행 코드
if __name__ == "__main__":
    # CSV 파일 경로
    file_path = "/content/drive/MyDrive/캡스톤/youtube_comments.csv"  # 입력 파일 경로
    print_preprocessed_data(file_path)


원본: ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ
전처리: 
--------------------------------------------------
원본: 하윤님은 언제나 사랑입니다❤
전처리: 하윤님은 언제나 사랑입니다
--------------------------------------------------
원본: 밍트융님 넘 좋아요. 화기애애하고 좋네요. 밍트융님이 행복할때는 제주도가서 김밥 먹을때 같아요. 
전처리: 밍트융님 넘 좋아요 화기애애하고 좋네요 밍트융님이 행복할때는 제주도가서 김밥 먹을때 같아요
--------------------------------------------------
원본: 우와..노랑윤이군요 ❤❤뽑아윤 무물타임 새로생겼군요. 😊
전처리: 우와노랑윤이군요 뽑아윤 무물타임 새로생겼군요
--------------------------------------------------
원본: 하윤님은 언제나 사랑입니다❤
전처리: 하윤님은 언제나 사랑입니다
--------------------------------------------------
원본: 유익한 정보 감사합니다.
전처리: 유익한 정보 감사합니다
--------------------------------------------------
원본: 하반기 채용공고 언제 올라와요? ㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠ
전처리: 하반기 채용공고 언제 올라와요
--------------------------------------------------
원본: 채용담당자분 볼지모르겠지만 올해 채용 없나요 답답해서 댓글 남겨봅니다 😢
전처리: 채용담당자분 볼지모르겠지만 올해 채용 없나요 답답해서 댓글 남겨봅니다
--------------------------------------------------
원본: 대한민국에서 가장 비싼땅이
울집이다...
전처리: 대한민국에서 가장 비싼땅이 울집이다
--------------------------------------------------
원본: 예쁘당❤
전처리: 예쁘당
-----

In [ ]:
file_path = "/content/drive/MyDrive/캡스톤/youtube_comments.csv"
analyze_emotions_with_reclassification('/content/drive/MyDrive/캡스톤/youtube_comments.csv')


TypeError: 'float' object is not iterable

In [105]:
#Original_Emotion	나오는 버
# 감정 분석 함수
def analyze_emotions_with_reclassification(file_path):
    # CSV 파일 읽기
    df = pd.read_csv(file_path)

    # "Comment" 열을 사용하여 문장 리스트 가져오기
    sentences = df['Comment'].tolist()
    predictions = []  # 감정 예측 결과를 저장할 리스트

    # 데이터 변환 및 감정 분석
    for sentence in sentences:
        data = [sentence, 0]  # 임시 라벨
        dataset = [data]

        test_dataset = BERTDataset(dataset, 0, 1, tokenizer, vocab, max_len, True, False)
        test_dataloader = DataLoader(test_dataset, batch_size=1, num_workers=0)

        model.eval()
        with torch.no_grad():
            for token_ids, valid_length, segment_ids, label in test_dataloader:
                token_ids = token_ids.long().to(device)
                segment_ids = segment_ids.long().to(device)
                out = model(token_ids, valid_length, segment_ids)
                logits = out.detach().cpu().numpy()
                pred_label = np.argmax(logits)
                predictions.append(pred_label)  # 예측 결과 추가

    # 기존 감정을 재분류
    reclassified_emotions = []
    for pred in predictions:
        if pred == 4:  # 행복 → Positive
            reclassified_emotions.append("Positive")
        elif pred in [2, 3]:  # 중립, 놀람 → Neutral
            reclassified_emotions.append("Neutral")
        elif pred in [0, 1, 5, 6]:  # 공포, 슬픔, 혐오, 분노 → Negative
            reclassified_emotions.append("Negative")

    # 재분류된 감정을 정수로 변환
    reclassified_emotions = [label_mapping[emotion] for emotion in reclassified_emotions]

    # 기존 데이터에 예측된 감정 추가
    df['Original_Emotion'] = [emotion_mapping[pred] for pred in predictions]  # 원래 7가지 감정
    df['Reclassified_Emotion'] = reclassified_emotions  # 재분류된 감정

    # 결과를 새로운 CSV 파일로 저장
    output_file = file_path.replace(".csv", "_with_emotions.csv")
    df.to_csv(output_file, index=False, encoding='utf-8-sig')

    print(f"분석 결과가 저장되었습니다: {output_file}")
    return output_file


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


도커 연결인데 모르겠움

In [108]:
!apt-get update
!apt-get install -y docker.io


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,224 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,620 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Pac

In [107]:
%mkdir /content/drive/MyDrive/emotion-analysis
%cd /content/drive/MyDrive/emotion-analysis


/content/drive/MyDrive/emotion-analysis


In [ ]:
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
import pandas as pd
import torch
from torch.utils.data import DataLoader

app = Flask(__name__)
CORS(app)

# 감정 분석 함수
def analyze_emotions_with_reclassification(file_path):
    df = pd.read_csv(file_path)
    sentences = df['Comment'].tolist()
    predictions = []

    for sentence in sentences:
        data = [sentence, 0]  # 임시 라벨
        dataset = [data]

        test_dataset = BERTDataset(dataset, 0, 1, tokenizer, vocab, max_len, True, False)
        test_dataloader = DataLoader(test_dataset, batch_size=1, num_workers=0)

        model.eval()
        with torch.no_grad():
            for token_ids, valid_length, segment_ids, label in test_dataloader:
                token_ids = token_ids.long().to(device)
                segment_ids = segment_ids.long().to(device)
                out = model(token_ids, valid_length, segment_ids)
                logits = out.detach().cpu().numpy()
                pred_label = np.argmax(logits)
                predictions.append(pred_label)

    reclassified_emotions = []
    for pred in predictions:
        if pred == 4:
            reclassified_emotions.append("Positive")
        elif pred in [2, 3]:
            reclassified_emotions.append("Neutral")
        elif pred in [0, 1, 5, 6]:
            reclassified_emotions.append("Negative")

    reclassified_emotions = [label_mapping[emotion] for emotion in reclassified_emotions]
    df['Reclassified_Emotion'] = reclassified_emotions

    output_file = file_path.replace(".csv", "_with_emotions.csv")
    df.to_csv(output_file, index=False, encoding='utf-8-sig')

    return output_file

@app.route('/analyze', methods=['POST'])
def upload_and_analyze():
    if 'file' not in request.files:
        return jsonify({"error": "No file uploaded"}), 400

    file = request.files['file']
    input_file = "./uploaded_file.csv"
    file.save(input_file)

    try:
        output_file = analyze_emotions_with_reclassification(input_file)
        return send_file(output_file, mimetype='text/csv', as_attachment=True, attachment_filename="output_with_emotions.csv")
    except Exception as e:
        return jsonify({"error": str(e)}), 500

if __name__ == "__main__":
    app.run(host='0.0.0.0', port=5000)


In [ ]:
app = Flask(__name__)
CORS(app)

# 업로드된 파일에서 감성 분석 수행
@app.route('/analyze', methods=['POST'])
def analyze():
    if 'file' not in request.files:
        return jsonify({'error': 'No file uploaded'}), 400

    file = request.files['file']
    file_path = "./uploaded_file.csv"
    file.save(file_path)

    try:
        # 감성 분석 수행
        output_file = analyze_emotions(file_path)
        return jsonify({'message': 'Emotion analysis completed.', 'output_file': output_file}), 200
    except Exception as e:
        return jsonify({'error': str(e)}), 500

# API 실행
if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)


In [ ]:
# 기존 데이터에 예측된 감정 추가
df['Original_Emotion'] = [emotion_mapping[pred] for pred in predictions]  # 원래 7가지 감정
df['Reclassified_Emotion'] = reclassified_emotions  # 재분류된 감정

# 결과를 새로운 CSV 파일로 저장
output_file = file_path.replace(".csv", "_with_emotions.csv")
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"분석 결과가 저장되었습니다: {output_file}")
return output_file


In [ ]:
#토큰화
tokenizer = KoBERTTokenizer.from_pretrained('skt/kobert-base-v1')
#tok = tokenizer.tokenize

def predict(predict_sentence):

    data = [predict_sentence, '0']
    dataset_another = [data]

    another_test = BERTDataset(dataset_another, 0, 1, tokenizer, vocab, max_len, True, False)
    test_dataloader = torch.utils.data.DataLoader(another_test, batch_size=batch_size, num_workers=5)

    model.eval()

    for batch_id, (token_ids, valid_length, segment_ids, label) in enumerate(test_dataloader):
        token_ids = token_ids.long().to(device)
        segment_ids = segment_ids.long().to(device)

        valid_length= valid_length
        label = label.long().to(device)

        out = model (token_ids, valid_length, segment_ids)

        test_eval=[]
        for i in out:
            logits=i
            logits = logits.detach().cpu().numpy()

            if np.argmax(logits) == 0:
              test_eval.append("공포")
            elif np.argmax(logits) == 1:
              test_eval.append("슬")
            elif np.argmax(logits) == 2:
              test_eval.append("중립")
            elif np.argmax(logits) == 3:
              test_eval.append("놀")
            elif np.argmax(logits) == 4:
              test_eval.append("행복")
            elif np.argmax(logits) == 5:
              test_eval.append("혐오")
            elif np.argmax(logits) == 6:
              test_eval.append("분노")

            print(test_eval[0] + " 하루를 보냈네요.")
#질문 무한반복하기! 0 입력시 종료
end = 1
while end == 1 :
    sentence = input("하고싶은 말을 입력해주세요 : ")
    if sentence == 0 :
        break
    predict(sentence)
    print("\n")

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.


하고싶은 말을 입력해주세요 : lx정보연구원 정말 유익한 정보를 주네요
행복 하루를 보냈네요.


하고싶은 말을 입력해주세요 : lx 이렇게 안 봤는데.. 내부 조사가 필요해보입니다
중립 하루를 보냈네요.


하고싶은 말을 입력해주세요 : 아니 니네는 세금만 받아 처먹고 이게 뭐하는짓이야
분노 하루를 보냈네요.


하고싶은 말을 입력해주세요 : 집에가고싶다
슬 하루를 보냈네요.


하고싶은 말을 입력해주세요 : 앞으로도 파이팅입니다
행복 하루를 보냈네요.




KeyboardInterrupt: Interrupted by user